## Make the GPT-2 simulate republican cases through Reinforcement Learning

In this homework, we will first train a reward model that assign higher reward to documents that sounds more like republican cases. Then we use RL to guide GPT-2 to complete the democratric cases in a republican way.

The reward model is covered in a previous notebook. All TODOs you need to finish lie in the RL part.

In [ ]:
# %pip install transformers trl

In [1]:
# load sc_cases_cleaned.pkl that we used in the previous notebooks
# can be also find in https://github.com/elliottash/nlp_lss_2023/blob/master/notebooks/sc_cases_cleaned.pkl
from google.colab import files
uploaded = files.upload()

Saving sc_cases_cleaned.pkl to sc_cases_cleaned.pkl


In [ ]:
# Pin versions compatible with the classic TRL PPO API used in this notebook,
# then HARD-restart the kernel so they actually load. Colab auto-reconnects.
!pip uninstall -y trl transformers tokenizers accelerate peft
!pip install "trl==0.8.6" "transformers==4.38.2" "tokenizers==0.15.2" "accelerate==0.27.2"
!pip show trl | grep -i version   # should print: Version: 0.8.6

# Force-restart the kernel (re-running cells in a live kernel keeps the OLD trl in memory).
import os; os.kill(os.getpid(), 9)

Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: tokenizers 0.15.2
Uninstalling tokenizers-0.15.2:
  Successfully uninstalled tokenizers-0.15.2
Found existing installation: accelerate 0.27.2
Uninstalling accelerate-0.27.2:
  Successfully uninstalled accelerate-0.27.2
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached accelerate-0.27.2-py3-none-any.whl.metadata (18 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
Using cac

In [1]:
import warnings; warnings.simplefilter('ignore')
import pandas as pd
import numpy as np

import torch
from tqdm import tqdm
import pandas as pd

tqdm.pandas()

from transformers import pipeline, AutoTokenizer, DistilBertTokenizerFast, DistilBertForSequenceClassification

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler


### train a classification model as our reward model

In [2]:

df = pd.read_pickle('sc_cases_cleaned.pkl', compression='gzip')
df = df.assign(author_id=(df['authorship']).astype('category').cat.codes)

# gpu or cpu?
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print (device)

model_name = 'distilbert-base-uncased' # huggingface model_ID or path to folder
model = DistilBertForSequenceClassification.from_pretrained(model_name)

tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
inputs = tokenizer(df['opinion_text'].tolist(), return_tensors="pt", padding=True, truncation=True)
labels = torch.tensor(df['x_republican'].tolist()).long()

optimizer = torch.optim.Adam([
    {'params': model.distilbert.parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3}
])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['opinion_text'].tolist(), df['x_republican'].tolist(), test_size=.2)

# generate batches
X_train, X_test, y_train, y_test = np.array(X_train[:608]), np.array(X_test[:152]), np.array(y_train[:608]), np.array(y_test[:152])
print (X_train.shape, X_test.shape, y_train.shape, y_test.shape)

X_train, X_test, y_train, y_test = X_train.reshape(-1, 8), X_test.reshape(-1, 8), y_train.reshape(-1, 8), y_test.reshape(-1, 8)
print (X_train.shape, X_test.shape, y_train.shape, y_test.shape)

X_train, X_test = X_train.tolist(), X_test.tolist()

# train
from tqdm import tqdm

model.to(device)
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for text, labels in tqdm(zip(X_train, y_train), total=len(X_train)):
        # prepare model input through our tokenizer
        model_inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256)
        # place everything on the right device
        model_inputs = {k:v.to(device) for k,v in model_inputs.items()}
        # labels have to be torch long tensors
        labels = torch.tensor(labels).long().to(device)
        # now, we can perform the forward pass
        output = model(**model_inputs, labels=labels)
        loss, logits = output[:2]
        # and the backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

torch.save(model, 'republican_classifier.pt')
republican_classifier = model

cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

(608,) (152,) (608,) (152,)
(76, 8) (19, 8) (76, 8) (19, 8)


100%|██████████| 76/76 [00:16<00:00,  4.60it/s]


In [3]:
from transformers import DistilBertTokenizer, TextClassificationPipeline

distil_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased', max_length=512, truncation=True)
pipeline = TextClassificationPipeline(model=republican_classifier, tokenizer=distil_tokenizer, return_all_scores=True, device=republican_classifier.device)

In [4]:
texts = df.loc[df['x_republican'] == 0, 'opinion_text'].tolist()
texts[0]

'JUSTICE GINSBURG delivered the opinion of the Court.\n\nPetitioner Remon Lee asserts that a Missouri trial court deprived him of due process when the court refused to grant an overnight continuance of his trial. Lee sought the continuance to locate subpoenaed, previously present, but suddenly missing witnesses key to his defense against felony charges. On direct review, the Missouri Court of Appeals disposed of the case on a state procedural ground. That court found the continuance motion defective under the State\'s rules. It therefore declined to consider the merits of Lee\'s plea that the trial court had denied him a fair opportunity to present a defense. Whether the state ground dispositive in the Missouri Court of Appeals is adequate to preclude federal habeas corpus review is the question we here consider and decide.\n\nOn the third day of his trial, Lee was convicted of first-degree murder and armed criminal action. His sole affirmative defense was an alibi; Lee maintained he w

In [5]:
pipeline(X_test[0][0], truncation=True)

[[{'label': 'LABEL_0', 'score': 0.999934196472168},
  {'label': 'LABEL_1', 'score': 6.584750371985137e-05}]]

### Reinforcement Learning

In [6]:
# prepare the dataset for reinforcement learning

from torch.utils.data import Dataset

class PPODataset(Dataset):
  def __init__(self, tokenizer, texts, input_min_text_length, input_max_text_length):
    self.tokenizer = tokenizer
    self.texts = texts
    self.random_sample = LengthSampler(input_min_text_length, input_max_text_length)

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, index):
    text = self.texts[index]
    sample = {}
    sample["input_ids"] = torch.tensor(tokenizer.encode(text)[: self.random_sample()])
    sample["query"] = tokenizer.decode(sample["input_ids"])
    return sample

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

dataset = PPODataset(tokenizer, texts, 10, 15)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# prepare arguments and configurations for RL

sent_kwargs = {"return_all_scores": True, "function_to_apply": "none", "batch_size": 16}

output_min_length = 50
output_max_length = 100
output_length_sampler = LengthSampler(output_min_length, output_max_length)


generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

def collator(data):
  return dict((key, [d[key] for d in data]) for key in data[0])

config = PPOConfig(
    model_name="gpt2",
    learning_rate=1.41e-5,
    batch_size=32,
    mini_batch_size=16,            # batch_size must be a multiple of mini_batch_size * gradient_accumulation_steps
    gradient_accumulation_steps=1,
)

In [9]:
# TODO: prepare model (use gpt2), reference model, and PPO trainer for RL

# prepare model (use gpt2), reference model, and PPO trainer for RL

# policy model with a value head (trained by PPO) and a frozen reference model (for the KL penalty)
model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)

ppo_trainer = PPOTrainer(
    config,
    model,
    ref_model,
    tokenizer,
    dataset=dataset,
    data_collator=collator,
)

# the trainer places the model on the accelerator; reuse that device in the cells below
device = ppo_trainer.accelerator.device
print("training on:", device)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/548M [00:00<?, ?B/s]

training on: cuda


In [10]:
# TODO: conduct PPO training loop here. For efficiency, you can just train 3 batches.

# conduct PPO training loop here. For efficiency, we just train 3 batches.

n_batches = 3
for step, batch in tqdm(enumerate(ppo_trainer.dataloader), total=n_batches):
    if step >= n_batches:
        break

    query_tensors = batch["input_ids"]

    # 1) generate a response from the policy model for every query in the batch
    response_tensors = []
    for query in query_tensors:
        gen_len = output_length_sampler()
        generation_kwargs["max_new_tokens"] = gen_len
        response = ppo_trainer.generate(query, **generation_kwargs)
        response_tensors.append(response.squeeze()[-gen_len:])
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    # 2) score each query+response with the reward model (prob of sounding "republican" = label 1)
    texts = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = pipeline(texts, **sent_kwargs)
    rewards = [torch.tensor(output[1]["score"]) for output in pipe_outputs]

    # 3) run a PPO optimization step and log
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)
    print(f"step {step}: mean reward = {torch.stack(rewards).mean().item():.4f}")

 33%|███▎      | 1/3 [00:36<01:13, 36.80s/it]

step 0: mean reward = -3.5878


 67%|██████▋   | 2/3 [01:14<00:37, 37.46s/it]

step 1: mean reward = -3.4256


100%|██████████| 3/3 [01:52<00:00, 37.44s/it]

step 2: mean reward = -3.5472


100%|██████████| 3/3 [01:52<00:00, 37.58s/it]


In [11]:
# visualize the outcomes generated by the RL tuned GPT-2

#### get a batch from the dataset
game_data = dict()
game_data["query"] = [dataset[i]["query"] for i in range(15)]
query_tensors = [dataset[i]["input_ids"] for i in range(15)]

response_tensors_ref, response_tensors = [], []
gen_kwargs = {"min_length": -1, "top_k": 0.0, "top_p": 1.0, "do_sample": True, "pad_token_id": tokenizer.eos_token_id}
#### get response from gpt2 and gpt2_ref
for i in range(15):
    gen_len = output_length_sampler()
    output = ref_model.generate(
        torch.tensor(query_tensors[i]).unsqueeze(dim=0).to(device), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()[-gen_len:]
    response_tensors_ref.append(output)
    output = model.generate(
        torch.tensor(query_tensors[i]).unsqueeze(dim=0).to(device), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()[-gen_len:]
    response_tensors.append(output)

#### decode responses
game_data["response (before)"] = [tokenizer.decode(response_tensors_ref[i]) for i in range(15)]
game_data["response (after)"] = [tokenizer.decode(response_tensors[i]) for i in range(15)]

#### sentiment analysis of query/response pairs before/after
texts = [q + r for q, r in zip(game_data["query"], game_data["response (before)"])]
game_data["rewards (before)"] = [output[1]["score"] for output in pipeline(texts, **sent_kwargs)]

texts = [q + r for q, r in zip(game_data["query"], game_data["response (after)"])]
game_data["rewards (after)"] = [output[1]["score"] for output in pipeline(texts, **sent_kwargs)]

# store results in a dataframe
df_results = pd.DataFrame(game_data)
df_results

,query,response (before),response (after),rewards (before),rewards (after)
0,JUSTICE GINSBURG delivered the opinion of,the Court. JOSEPH IOWA delivered the opinion ...,the Court of Appeal. The judgment of the Stat...,-3.838808,-3.418543
1,Justice Ginsburg delivered the opinion of the ...,"\nII\n\nThe 1971 decision, 42 U.S.C. § 1401b(a...","\nJustices\n\nIn Vella v. Texas, PC ULP 48 the...",-3.994941,-3.861309
2,JUSTICE BREYER delivered the opinion of the Court,. SCALIA delivered the opinion of the Court.\n...,"of the United States of America, No. 6NS&J, N...",-2.857014,-3.124022
3,JUSTICE GINSBURG delivered the opinion of the ...,"the Court.\n\nMARWORN, JUSTICE WHITE, and BLA...","the Court of Appeals, Nevada.\n\nJustices: CO...",-3.959379,-4.079569
4,Justice Breyer delivered the opinion of the Co...,The Court also held that petitioners are entit...,{226} 26 FR 658 888 (1966); The Telephone and ...,-3.415958,-3.101505
5,JUSTICE GINSBURG delivered the opinion of the,"of the Court BURGER, J., for the 5-4 majority...",of the Rocky Mountain United Club not simply ...,-3.276056,-3.442984
6,Justice Breyer delivered the opinion of the Co...,case starts with the indictment of the Reichs...,law is identical to several other laws govern...,-1.557212,-3.200941
7,Justice Breyer delivered the opinion of the Co...,"I agree with ^.* in Barton, which declares th...",____________ The judgment of the Court of the ...,-3.191651,-3.336360
8,Justice Breyer delivered the opinion of the Co...,_______________\n\nJUSTICE BLACKMUN delivered ...,_____________________________ Clerks 1662*160 ...,-2.955317,-1.613287
9,Justice Breyer delivered the opinion of the Co...,"opened by *\n\nJeffrey G. Stone. GrowR, LLC, t...",year 2017-18-0583\n\nMichael G. U take Charge...,-3.400977,-2.541341


In [12]:
print("mean:")
display(df_results[["rewards (before)", "rewards (after)"]].mean())
print()
print("median:")
display(df_results[["rewards (before)", "rewards (after)"]].median())

mean:


,0
rewards (before),-3.331421
rewards (after),-3.276062



median:


,0
rewards (before),-3.415958
rewards (after),-3.366691
